> **Disclaimer:** This notebook is for *educational and research training purposes only*. It does not constitute medical, clinical, or diagnostic advice. All computational predictions are approximations from machine learning models and must not be treated as experimental results. See the full [DISCLAIMER](https://github.com/JobAiReady/lagos-bio-design/blob/main/DISCLAIMER.md) and [PRIVACY POLICY](https://github.com/JobAiReady/lagos-bio-design/blob/main/PRIVACY_POLICY.md).
>
> **Attribution:** This notebook uses [BioEmu-1](https://github.com/microsoft/bioemu) by Microsoft Research (Lewis et al., *Science* 2025), licensed under [MIT](https://github.com/microsoft/bioemu/blob/main/LICENSE). Model weights from [microsoft/bioemu](https://huggingface.co/microsoft/bioemu) on HuggingFace.

# Module 2 Bonus: Protein Stability Validation with BioEmu-1
## Lab: From Static Snapshot to Conformational Ensemble

**Lagos Bio-Design Bootcamp** | [JobAiReady](https://github.com/JobAiReady/lagos-bio-design)

---

### Objective
Generate a **conformational ensemble** for your Module 2 designed protein and determine whether it is conformationally **stable** — not just whether it folds, but whether it *stays* folded.

### Prerequisites
- Completed Module 2 (you have a designed protein sequence)
- **GPU Runtime enabled** (Runtime → Change runtime type → T4 GPU)
- ~10–15 minutes for first-time setup (model weights ~3.5 GB)

### Deliverable
An ensemble analysis showing your design's stability profile, compared against an unstable control peptide. You'll get a clear verdict: **STABLE**, **MARGINALLY STABLE**, or **UNSTABLE**.

> ⚠️ **GPU Required.** Go to *Runtime → Change runtime type → T4 GPU* before running.

---
## Background: Why Static Validation Isn't Enough

In Module 2, you validated your design with a **self-consistency check** — fold the sequence with ESMFold, compare to the intended RFDiffusion backbone, confirm RMSD < 2 Å. You proved your design *can* fold. But **can it *stay* folded**?

### The Real-World Failure Case

In early protein design campaigns (2018–2022), many computationally designed proteins had excellent predicted metrics — high pLDDT (>90), low RMSD (<1 Å) — yet **failed when synthesized in the lab**. They aggregated, misfolded, or existed as molten globules (proteins that have a hydrophobic core but no fixed tertiary structure). The missing piece: structure prediction tools only predicted the *lowest-energy snapshot*, not whether the protein actually *stays there* under thermal motion.

This is why pharmaceutical companies historically had to synthesize hundreds of computational designs to find one that actually worked at the bench. The bottleneck wasn't generating designs — it was filtering out the unstable ones.

### The Analogy

Think of it this way:

- **AlphaFold/ESMFold** gives you a **photograph** of your protein at its best moment — one snapshot, perfectly posed.
- **BioEmu-1** gives you the **security camera footage** — showing everything the protein does when you're not watching.

A *stable* design looks the same in every frame. An *unstable* one is caught flickering between folded and unfolded poses.

### How BioEmu-1 Works

BioEmu-1 (Lewis et al., *Science* 2025) is a **diffusion model trained on thousands of molecular dynamics (MD) simulations**. Given only a sequence, it generates statistically independent conformational samples drawn from the protein's **equilibrium distribution** — the population of shapes the protein occupies in thermal equilibrium. It does in hours what would take traditional MD simulations *years* of compute time.

### Key Terms

| Term | Definition |
|------|-----------|
| **Conformational Ensemble** | The full set of 3D shapes a protein adopts due to thermal motion, weighted by how often each occurs |
| **Equilibrium Distribution** | The probability distribution over conformations when the protein is at thermal equilibrium (Boltzmann-weighted) |
| **Free Energy Landscape** | The mapping of conformations to energies. Stable proteins sit in deep, narrow energy wells; unstable ones in shallow, broad ones |
| **Molecular Dynamics (MD)** | Physics-based simulation of atomic motion over time. The traditional gold standard, but extremely expensive (years of GPU time for milliseconds of biology) |
| **Protein Stability** | The thermodynamic preference for the folded state. Quantified by free energy of folding (ΔG_fold) — more negative = more stable |
| **Steering (SMC)** | Sequential Monte Carlo sampling that BioEmu uses to bias generation toward physically plausible structures (no clashes, no broken chains) |

---
## 🤔 Prediction Step (Before You Run Anything)

**Stop and write down your prediction below.** This is your hypothesis — we'll test it against the data in a few cells.

1. Do you think your Module 2 design will be **stable** (tight ensemble, all conformations look the same) or **unstable** (diverse ensemble, lots of different shapes)?
2. *Why?* What features of your designed sequence make you think so? Consider:
   - Is the hydrophobic core well-packed?
   - Are there charged residues paired with opposite charges?
   - Is the secondary structure regular (helices, sheets) or loopy?
3. How confident are you (1–10)?

**Write your answer in this cell** (double-click to edit) before continuing:

> *My prediction: ___________________________*

In [ ]:
# Step 0: Setup — Verify GPU and install BioEmu
#
# **What's happening:** We're installing BioEmu as a single pip package. Unlike RFDiffusion
# (which required cloning a repo and installing many transitive dependencies), BioEmu bundles
# everything — including its own MSA retrieval (via ColabFold) — into one install.
# Model weights (~3.5 GB) auto-download from HuggingFace on first run.

import torch
assert torch.cuda.is_available(), 'GPU not found! Go to Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')

# Install BioEmu with CUDA support
!pip install -q bioemu[cuda]

# Verify import
from bioemu.sample import main as bioemu_sample
print('\n✓ BioEmu installed and ready.')

In [ ]:
# Step 1: Define your two test sequences
#
# **What's happening:** We're setting up a controlled experiment — your designed protein vs.
# a known minimally-stable peptide. Without a control, you can't tell if a 'diverse ensemble'
# means your design is bad or just normal protein flexibility. The control gives you a baseline.

# (A) YOUR designed protein from Module 2.
# Replace this with the sequence from your Module 2 ProteinMPNN output.
# Fallback: a reference 60-residue de novo helical bundle (well-designed, expected to be stable)
designed_seq = (
    'MKQLEDKVEELLSKNYHLENEVARLKKLVGER'
    'MKQLEDKVEELLSKNYHLENEVARLKKLVGER'
)[:60]  # 60 residues

# (B) Chignolin — a 10-residue beta-hairpin known to be MARGINALLY STABLE.
# It spends significant time in both folded and unfolded states.
# This is our 'unstable' baseline for comparison.
control_seq = 'GYDPETGTWG'

print(f'(A) Your design       : {designed_seq}  ({len(designed_seq)} residues)')
print(f'(B) Chignolin control : {control_seq}  ({len(control_seq)} residues)')
print('\nWe will generate 50 conformational samples of each and compare their ensembles.')

In [ ]:
# Step 2: Generate the conformational ensemble for your DESIGNED protein
#
# **What's happening:** BioEmu is running its diffusion model 50 independent times. Each run
# starts from random noise and denoises into a complete protein structure — but conditioned
# on the sequence so that all 50 outputs are valid conformations of THE SAME protein, drawn
# from its equilibrium distribution. Think of each sample as 'one frame of security camera footage.'
#
# Sampling time on T4 for 60 residues: ~5-8 minutes for 50 samples.

import os, shutil

designed_dir = '/content/ensemble_designed'
if os.path.exists(designed_dir):
    shutil.rmtree(designed_dir)

bioemu_sample(
    sequence=designed_seq,
    num_samples=50,
    output_dir=designed_dir,
    filter_samples=True,
)

# Count how many physically valid samples survived filtering
import glob
designed_pdbs = sorted(glob.glob(os.path.join(designed_dir, '**', '*.pdb'), recursive=True))
print(f'\n✓ Generated {len(designed_pdbs)} valid conformational samples for your design.')

In [ ]:
# Step 3: Generate the ensemble for the CHIGNOLIN control
#
# **What's happening:** Same process for Chignolin — a tiny 10-residue beta-hairpin that
# is known experimentally to be marginally stable. It exists in equilibrium between folded
# and unfolded states. Comparing your design's ensemble to Chignolin's gives you a concrete
# reference for what 'unstable' looks like.
#
# This is fast — ~1-2 minutes for 50 samples since the peptide is tiny.

control_dir = '/content/ensemble_control'
if os.path.exists(control_dir):
    shutil.rmtree(control_dir)

bioemu_sample(
    sequence=control_seq,
    num_samples=50,
    output_dir=control_dir,
    filter_samples=True,
)

control_pdbs = sorted(glob.glob(os.path.join(control_dir, '**', '*.pdb'), recursive=True))
print(f'\n✓ Generated {len(control_pdbs)} valid conformational samples for Chignolin control.')

In [ ]:
# Step 4: Analyze — compute pairwise CA-RMSD distributions for both ensembles
#
# **What's happening:** For each ensemble, we compute the RMSD between every pair of
# conformations. A tight ensemble (stable protein) has low pairwise RMSDs — all the frames
# look alike. A loose ensemble (unstable protein) has high pairwise RMSDs — the frames
# are all different shapes.

import numpy as np
import matplotlib.pyplot as plt
from Bio.PDB import PDBParser, Superimposer
import warnings
warnings.filterwarnings('ignore')

def get_ca_atoms(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('s', pdb_path)
    return [a for a in structure.get_atoms() if a.get_name() == 'CA']

def pairwise_rmsd_matrix(pdb_files):
    n = len(pdb_files)
    rmsds = np.zeros((n, n))
    ca_lists = [get_ca_atoms(f) for f in pdb_files]
    sup = Superimposer()
    for i in range(n):
        for j in range(i+1, n):
            if len(ca_lists[i]) != len(ca_lists[j]):
                continue
            sup.set_atoms(ca_lists[i], ca_lists[j])
            rmsds[i, j] = rmsds[j, i] = sup.rms
    return rmsds

rmsd_designed = pairwise_rmsd_matrix(designed_pdbs)
rmsd_control = pairwise_rmsd_matrix(control_pdbs)

# Extract upper triangle (unique pairs)
design_pairs = rmsd_designed[np.triu_indices_from(rmsd_designed, k=1)]
control_pairs = rmsd_control[np.triu_indices_from(rmsd_control, k=1)]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(design_pairs, bins=30, alpha=0.6, label=f'Your design (mean={design_pairs.mean():.2f}Å)', color='steelblue')
ax.hist(control_pairs, bins=30, alpha=0.6, label=f'Chignolin control (mean={control_pairs.mean():.2f}Å)', color='salmon')
ax.set_xlabel('Pairwise CA-RMSD (Å)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Conformational Ensemble Diversity', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nYour design  — mean pairwise RMSD: {design_pairs.mean():.2f} Å (lower = more stable)')
print(f'Chignolin    — mean pairwise RMSD: {control_pairs.mean():.2f} Å')
print('\n💭 **Think about it:** Which distribution is tighter? What does the width of each tell you?')

In [ ]:
# Step 5: Cluster the conformational states
#
# **What's happening:** We use hierarchical clustering with a 2.0 Å RMSD cutoff to group
# similar conformations together. The fraction of samples in the LARGEST cluster tells us
# how dominant the most-populated state is. A protein with 95% of its ensemble in one
# cluster has a single dominant fold — i.e., it's stable.

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

def dominant_cluster_fraction(rmsd_matrix, cutoff=2.0):
    if rmsd_matrix.shape[0] < 2:
        return 1.0, 1
    condensed = squareform(rmsd_matrix, checks=False)
    Z = linkage(condensed, method='average')
    clusters = fcluster(Z, t=cutoff, criterion='distance')
    counts = np.bincount(clusters)[1:]
    return counts.max() / counts.sum(), len(counts)

design_dom_frac, design_n_clusters = dominant_cluster_fraction(rmsd_designed, cutoff=2.0)
control_dom_frac, control_n_clusters = dominant_cluster_fraction(rmsd_control, cutoff=2.0)

print(f'YOUR DESIGN:')
print(f'  Number of distinct conformational states (clusters): {design_n_clusters}')
print(f'  Fraction of ensemble in dominant state             : {design_dom_frac:.1%}')
print(f'\nCHIGNOLIN CONTROL:')
print(f'  Number of distinct conformational states (clusters): {control_n_clusters}')
print(f'  Fraction of ensemble in dominant state             : {control_dom_frac:.1%}')
print('\n💭 **Think about it:** If 95% of your ensemble falls in one cluster, your protein')
print('strongly prefers a single fold. If it\'s 50/50, it spends equal time in two states —')
print('a potential red flag for instability.')

In [ ]:
# Step 6: Visualize — overlay 5 conformations from each ensemble
#
# **What's happening:** We're overlaying multiple conformational samples to visually assess
# rigidity. A stable protein's traces stack nearly on top of each other — like a stack of
# transparencies of the same drawing. An unstable one looks like spaghetti — every sample
# is a different shape.

import py3Dmol

def overlay_view(pdb_files, color, label, n=5):
    view = py3Dmol.view(width=400, height=400)
    for pdb in pdb_files[:n]:
        with open(pdb) as f:
            view.addModel(f.read(), 'pdb')
    view.setStyle({'cartoon': {'color': color, 'opacity': 0.5}})
    view.zoomTo()
    return view

print('LEFT: Your design (blue overlay) — should look RIGID if stable')
v1 = overlay_view(designed_pdbs, 'steelblue', 'Your design')
v1.show()

print('\nRIGHT: Chignolin control (red overlay) — known to be flexible/marginal')
v2 = overlay_view(control_pdbs, 'salmon', 'Chignolin')
v2.show()

In [ ]:
# Step 7: The Verdict — is your design stable?
#
# We use the dominant-cluster fraction as a stability metric:
#   >80%  →  STABLE             (single dominant fold; ready for synthesis)
#   60-80% →  MARGINALLY STABLE (some fluctuation; consider redesign)
#   <60%  →  UNSTABLE           (multiple competing states; do NOT synthesize)

def verdict(frac):
    if frac > 0.80:
        return '✅ STABLE', 'green'
    elif frac > 0.60:
        return '⚠️  MARGINALLY STABLE', 'orange'
    else:
        return '❌ UNSTABLE', 'red'

design_verdict, _ = verdict(design_dom_frac)
control_verdict, _ = verdict(control_dom_frac)

print('=' * 60)
print(f'FINAL VERDICT')
print('=' * 60)
print(f'Your designed protein  : {design_verdict}  ({design_dom_frac:.0%} in dominant state)')
print(f'Chignolin control      : {control_verdict}  ({control_dom_frac:.0%} in dominant state)')
print('=' * 60)
print('\n📝 Now compare to your prediction in the Prediction Step.')
print('   Did the result match your hypothesis?')

---
## Reflection Questions

Take a few minutes to answer these in your own words:

1. **Was your prediction correct?** If not, what surprised you about the result?

2. **Confidence shift:** How does this stability analysis change your confidence in your Module 2 design? Would you recommend it for DNA synthesis?

3. **If your protein showed instability**, what modifications might improve it? *Hints:*
   - Better hydrophobic core packing (bulky non-polar residues like Leu, Ile, Val)
   - Salt bridges (paired Lys/Arg + Glu/Asp)
   - Disulfide bonds (Cys pairs at the right distance)
   - Avoiding helix-breakers (Pro, Gly) in the middle of helices

4. **Why is this analysis impossible with AlphaFold/ESMFold alone?** What information does an ensemble give you that a single static prediction cannot?

5. **Connection to Module 4:** The Lassa GPC undergoes conformational changes during viral entry. How might you use BioEmu-1 to validate that a designed binder works across the GPC's full conformational ensemble — not just one snapshot?

---
## Summary: The Extended Pipeline

Before this lab, your design pipeline answered one question: *Does it fold?* Now it answers two:

```
  RFDiffusion  →  ProteinMPNN  →  ESMFold        →  BioEmu-1
   (generate)      (sequence)      (does it fold?)   (does it stay folded?)
    backbone        design          RMSD < 2 Å        >80% in dominant state
    from noise      for fold        ✓/✗               STABLE / UNSTABLE
```

### What you learned

1. **Static validation isn't enough** — high pLDDT and low RMSD don't guarantee a design will work in the lab
2. **Conformational ensembles** reveal the *distribution* of shapes, not just the most likely one
3. **A control experiment** (Chignolin) gives you a concrete reference point for 'unstable'
4. **BioEmu-1 is fast** — it does in hours what traditional MD takes years to do
5. **Stability is now a screenable property** — you can filter unstable designs *before* synthesis, saving thousands of dollars

### Connection to the rest of the bootcamp

- **Module 3 (VibeGen):** Designs proteins for *target dynamics*. BioEmu validates that the designed motions actually emerge.
- **Module 4 (Lassa Capstone):** Use BioEmu to check that your binder candidates are stable AND that they work across the GPC's conformational ensemble.
- **Module 5 (Biosecurity):** Stability prediction can also flag designs with unusual thermodynamic profiles for additional review.

### References

- Lewis, S., Hempel, T., Jiménez-Luna, J., et al. "Scalable emulation of protein equilibrium ensembles with generative deep learning." *Science* (2025). [DOI](https://www.science.org/doi/10.1126/science.adv9817)
- [GitHub: microsoft/bioemu](https://github.com/microsoft/bioemu) (MIT license)
- [HuggingFace: microsoft/bioemu](https://huggingface.co/microsoft/bioemu) (model weights)
- [Microsoft Research blog post on BioEmu-1](https://www.microsoft.com/en-us/research/blog/exploring-the-structural-changes-driving-protein-function-with-bioemu-1/)

**Next:** Module 3 — Generative AI: Hallucination as a Feature (with VibeGen Bonus Lab)